In [ ]:
import torch
from torch.distributions import Dirichlet,Categorical
from torch.distributions.kl import kl_divergence
from tqdm.auto import trange
import matplotlib.pyplot as plt
from torch.distributions import Uniform
from wordcloud import WordCloud

# LDA Example

In [ ]:
from sklearn.datasets import fetch_20newsgroups
dataset = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

In [ ]:
len(dataset['data'])

In [ ]:
import re
import spacy
from sklearn.feature_extraction.text import CountVectorizer

nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

def clean_and_lemmatize(text):
    text = re.sub(r'\S*@\S*\s?', '', text)  # Remove emails
    text = re.sub(r'\s+', ' ', text)       # Remove newline chars
    text = re.sub(r"\'", "", text)         # Remove single quotes
    
    doc = nlp(text)
    
    # only keep nouns, adjectives, verbs, and adverbs using spacy
    allowed_postags = ['NOUN', 'ADJ', 'VERB', 'ADV']
    tokens = [
        token.lemma_ for token in doc 
        if token.pos_ in allowed_postags 
        and not token.is_stop 
        and len(token.lemma_) > 2 # Drop 1-2 character artifacts
    ]
    
    return " ".join(tokens)

In [ ]:
# clean first 5000 documents
cleaned_docs = [clean_and_lemmatize(doc) for doc in dataset['data'][:5000]]

vectorizer = CountVectorizer(
    max_df=0.5,        # Drop words in > 50% of docs
    min_df=10,         # Drop words in < 10 docs
    max_features=5000  # Keep top 5000 terms
)

# takes ~1 minute to run with 5000 docs
document_term_matrix = vectorizer.fit_transform(cleaned_docs)
doc_word_array = torch.from_numpy(document_term_matrix.toarray())

In [ ]:
vectorizer.get_feature_names_out()

In [ ]:
def cavi_step(word_count_matrix, g_theta, g_phi, g_z, alpha, beta):
    
    # g_theta: D x K
    # g_phi: K x V
    # g_z: D x V x K
    # word_count_matrix: D x V

    # update g_z
    e_log_theta = torch.digamma(g_theta.concentration) - torch.digamma(g_theta.concentration.sum(axis = 1,keepdim = True))
    e_log_phi = torch.digamma(g_phi.concentration) - torch.digamma(g_phi.concentration.sum(axis = 1,keepdim = True))
    # D x 1 x K
    # 1 x V x K
    g_z = Categorical(logits = e_log_theta.unsqueeze(1) + e_log_phi.T.unsqueeze(0))

    # update g_theta
    theta_concentration = alpha + torch.sum(word_count_matrix.unsqueeze(2) * g_z.probs, axis = 1)
    g_theta = Dirichlet(theta_concentration)

    # update g_phi
    phi_concentration = beta + torch.sum(word_count_matrix.unsqueeze(2) * g_z.probs, axis = 0).T
    g_phi = Dirichlet(phi_concentration)

    return g_theta, g_phi, g_z
    

In [ ]:
def elbo(word_count_matrix, g_theta, g_phi, g_z, p_theta, p_phi):
    
    e_log_theta = torch.digamma(g_theta.concentration) - torch.digamma(g_theta.concentration.sum(axis = 1,keepdim = True))
    e_log_phi = torch.digamma(g_phi.concentration) - torch.digamma(g_phi.concentration.sum(axis = 1,keepdim = True))
    expected_log_joint = e_log_theta.unsqueeze(1) + e_log_phi.T.unsqueeze(0)

    log_g_z = torch.log(g_z.probs)

    term_1 = (word_count_matrix.unsqueeze(2) * g_z.probs * (expected_log_joint - log_g_z)).sum()
    
    term_2 = -kl_divergence(g_theta,p_theta).sum()
    term_3 = -kl_divergence(g_phi,p_phi).sum()

    elbo = term_1 + term_2 + term_3

    return elbo / torch.sum(word_count_matrix)

In [ ]:
def cavi(word_count_matrix, 
         num_topics=20, 
         num_iters=100,
         seed=1234
        ):
    torch.manual_seed(seed)
    word_count_matrix = word_count_matrix.float()
    D, V = word_count_matrix.shape
    K = num_topics    
    alpha = torch.ones(K)
    beta = torch.ones(V)
    
    # Initialize the variational posteriors.
    g_theta = Dirichlet(Uniform(1, 5).sample((D, K)))
    g_phi = Dirichlet(Uniform(1, 5).sample((K, V)))
    g_z = Categorical(logits=torch.zeros((D, V, K)))

    p_theta = Dirichlet(alpha)
    p_phi = Dirichlet(beta)
    
    # Run CAVI
    elbos = [elbo(word_count_matrix, g_theta, g_phi, g_z, p_theta, p_phi)]
    for itr in trange(num_iters):
        g_theta, g_phi, g_z = cavi_step(
            word_count_matrix, 
            g_theta,
            g_phi,
            g_z,
            alpha,
            beta
        )
        
        elbos.append(elbo(word_count_matrix, g_theta, g_phi, g_z, p_theta, p_phi))
    return torch.tensor(elbos), (g_theta, g_phi, g_z)

In [ ]:
elbos, (g_theta_star, g_phi_star, g_z_star) = cavi(
    doc_word_array,
    num_iters = 200
)

In [ ]:
fig,ax = plt.subplots(1,1,figsize = (10,6))
ax.plot(
    torch.arange(0,201),
    elbos
)
ax.set_title("Averaged ELBO by Iteration of CAVI")
ax.set_xlabel("Iteration")
ax.set_ylabel("Averaged ELBO Value")
fig.savefig("elbo_plot.png",dpi=300,bbox_inches='tight')

## Inference

In [ ]:
vocabulary = vectorizer.get_feature_names_out()
print("Top 10 Words for each Latent Topic")
for i in range(20):
    top_10_indices = torch.argsort(g_phi_star.mean[i,:],descending=True)[:10]
    print(f"Topic {i+1} Words: {'\t'.join(vocabulary[top_10_indices])}")

In [ ]:
# topic 12, 6
top_100_words_t6 = vocabulary[torch.argsort(g_phi_star.mean[5,:],descending=True)[:100]]
top_100_probs_t6 = torch.sort(g_phi_star.mean[5,:],descending=True)[:100][0]

top_100_words_t12 = vocabulary[torch.argsort(g_phi_star.mean[11,:],descending=True)[:100]]
top_100_probs_t12 = torch.sort(g_phi_star.mean[11,:],descending=True)[:100][0]

probs_t6_dict = {
    word:prob.item() for word,prob in zip(top_100_words_t6,top_100_probs_t6)
}
probs_t12_dict = {
    word:prob.item() for word,prob in zip(top_100_words_t12,top_100_probs_t12)
}

In [ ]:
wc_t6 = WordCloud(width=800, height=400, background_color="white").generate_from_frequencies(probs_t6_dict)
wc_t12 = WordCloud(width=800, height=400, background_color="white").generate_from_frequencies(probs_t12_dict)

fig, axes = plt.subplots(1, 2, figsize=(12, 8))

axes[0].imshow(wc_t6, interpolation="bilinear")
axes[0].set_title("Topic 6 Top 100 Words")
axes[0].axis("off")

axes[1].imshow(wc_t12, interpolation="bilinear")
axes[1].set_title("Topic 12 Top 100 Words")
axes[1].axis("off")

fig.tight_layout()
fig.show()
fig.savefig("word_clouds.png",dpi=300,bbox_inches='tight')

In [ ]:
true_topics = dataset['target'][:5000]
topic_labels = dataset['target_names']
topic_label_terms = [topic_labels[i] for i in true_topics]
space_article_indices = [i for i,j in enumerate(topic_label_terms) if j == 'sci.space']
hockey_article_indices = [i for i,j in enumerate(topic_label_terms) if j == 'rec.sport.hockey']

In [ ]:
topic_labels

In [ ]:
len(hockey_article_indices)

In [ ]:
len(space_article_indices)

In [ ]:
topics, counts = torch.unique(
    torch.argmax(g_theta_star.mean[hockey_article_indices,:],axis = 1),
    return_counts = True
)
print(topics)
print(counts/counts.sum())

In [ ]:
topics, counts = torch.unique(
    torch.argmax(g_theta_star.mean[space_article_indices,:],axis = 1),
    return_counts = True
)
print(topics)
print(counts/counts.sum())